# 3 — Inference: the native point head over the whole Brackish val split

This branch runs `cropcounter`'s **native point task** — heat branch only, no `wh`/`off`
geometry — on the *same* CFD Brackish val frames a parked branch trained a **box** head on.
The question the two notebooks after this one answer is narrow and worth stating before any
number appears:

> Does dropping the box head's geometry branch cost the heat branch anything, or free capacity?

This notebook only *produces* predictions; `4_evaluate.ipynb` scores them. Everything is
written in **one file shape** — CVAT-for-images 1.1 points — so the point head, the box head's
box centres and the released RF-DETR-Nano's box centres all reach the scorer as the same kind
of object and are judged by the same function (Liam's verbatim `match_image`, in
`examples/FishDetection/scripts/point_in_box.py`).

**What this notebook writes to Drive** (`results/points/`):

| File | What it is |
| :-- | :-- |
| `pred_points_best.xml`, `pred_points_last.xml` | the point head, NMS 1.5, decoded at a **low floor τ = 0.05** |
| `pred_points_best_nms5.xml`, `pred_points_last_nms5.xml` | the same at NMS 5, for the record |
| `pred_boxhead_best_centres.xml`, `pred_boxhead_last_centres.xml` | the parked box run's boxes reduced to centres |
| `pred_rfdetr_nano_centres.xml` | the released detector's boxes reduced to centres |
| `counts_best.csv`, `counts_last.csv` | `image_name, predicted_count` at the **calibrated** τ |
| `flops_params.json` | measured trainable params and measured GFLOPs |
| `viz/` | overlays on a seeded handful of val frames |

**Why the floor τ = 0.05 matters.** The XMLs are decoded once at 0.05 and never again.
`decode_peaks` applies NMS *before* thresholding and NMS ranks by score, so the set of points
kept at any higher τ is exactly a mask over the 0.05 set — which is what lets `4_evaluate`
sweep the whole threshold grid without a second forward pass. Decoding at the operating point
instead would make the sweep impossible.

> **Runtime**: A100. Run order is `1_reformat` → `2_training` → **`3_inference`** → `4_evaluate`.
> Colab gives each notebook its own VM, so cell 1b re-creates the data if this one lands fresh.

## 1 · Drive, paths, code

Mounts Drive, fixes the paths every cell below uses, clones this branch and installs it
editable. Safe to re-run: the clone is wiped and redone each time.

Two data roots, and the distinction is load-bearing (see `examples/FishDetection/README.md`):
`DATA` is the COCO **bbox** subset — the box ground truth `4_evaluate` scores against — and
`POINTS` is the separate **keypoints** root the point model trains and predicts on, with
`cfd_id_map.json` beside it carrying the CFD-string-id → points-int-id join.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib, json, os, sys, time
from pathlib import Path

DRIVE  = '/content/drive/MyDrive/frozen-trunk-detection'
REPO   = '/content/crop-counter'
DATA   = '/content/data/brackish'          # COCO bbox subset  -> the GT boxes
POINTS = '/content/data/brackish_points'   # COCO keypoints root + cfd_id_map.json
CFD    = '/content/cfd'
RUN    = f'{DRIVE}/runs/brackish_points_s0'
RES    = f'{DRIVE}/results/points'
for sub in (RES, f'{RES}/figures', f'{RES}/viz', CFD):
    os.makedirs(sub, exist_ok=True)
print('Drive:', sorted(os.listdir(DRIVE)))
print('run  :', RUN, '| exists', os.path.isdir(RUN))

%cd /content
!rm -rf crop-counter
!git clone --branch poc/fish-points --depth 1 https://github.com/InsightML/crop-counter.git
%cd /content/crop-counter
!git log --oneline -3
# torch/torchvision come with the Colab image. torchmetrics + termcolor are needed only so
# Meta's dinov3 hubconf imports; ijson (the [cfd] extra) streams the 1.9M-record CFD metadata.
!pip install -q -e ".[dev,portal,cfd]"
# A running kernel does not re-read site-packages' .pth files, so the editable install is
# invisible to THIS process until a restart (subprocess `!python -m cropcounter...` sees it).
for path in ('/content/crop-counter/src', '/content/crop-counter/examples/FishDetection/scripts'):
    if path not in sys.path:
        sys.path.insert(0, path)
importlib.invalidate_caches()

# The frozen backbone is gated and lives on Drive; symlink rather than copy (1 x 350 MB).
os.makedirs(f'{REPO}/weights', exist_ok=True)
BACKBONE = 'dinov3_convnext_base_pretrain_lvd1689m-801f2ba9.pth'
link, target = f'{REPO}/weights/{BACKBONE}', f'{DRIVE}/weights/{BACKBONE}'
if not os.path.exists(link):
    os.symlink(target, link)

import torch
import cropcounter
import nb_helpers as nbh
import point_in_box as pib
print('cropcounter', cropcounter.__file__)
print('nb_helpers ', nbh.__file__)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

### 1b · Ensure the Brackish slice and the points root are on this VM

Idempotent, and present in **every** notebook of this example for the same reason: Colab
attaches each notebook to its own runtime, so the frames `1_reformat` fetched are not here
unless this notebook is sharing that session. Metadata is 47 MB; ~14.7k frames from the LILA
GCS mirror take ~4–5 min.

In [ ]:
have_points = os.path.exists(f'{POINTS}/val/annotations.json')
have_images = os.path.isdir(f'{DATA}/val/images') and len(os.listdir(f'{DATA}/val/images')) > 0
if not (have_points and have_images):
    META = f'{CFD}/community_fish_detection_dataset.json.zip'
    if not os.path.exists(META):
        !wget -q -O {META} https://lilawildlife.blob.core.windows.net/lila-wildlife/community-fish-detection-dataset/community_fish_detection_dataset.json.zip
    !python -m cropcounter.cfd subset --metadata {META} --out {DATA} --sources brackish_dataset --train-cap 100000 --val-cap 100000 --seed 0 --no-progress
    !python -m cropcounter.cfd fetch  --subset {DATA} --max-side 1024 --workers 32 --mirror gcs --no-progress
    !python -m cropcounter.cfd points --subset {DATA} --out {POINTS}

for split in ('train', 'val'):
    n_files = len(os.listdir(f'{DATA}/{split}/images'))
    doc = json.load(open(f'{POINTS}/{split}/annotations.json'))
    print(f'{split}: {n_files} image files | {len(doc["images"])} point-root image records | '
          f'{len(doc["annotations"])} points')
print('id map splits:', list(json.load(open(f'{POINTS}/cfd_id_map.json'))))

### 1c · Machine

bf16 autocast (what `predict_prob` uses on CUDA) needs compute capability ≥ 8. A T4 is 7.5 and
would silently run fp32 at a different speed and slightly different numerics, so this asserts
rather than warns.

In [ ]:
assert torch.cuda.is_available(), 'no GPU — set Runtime > Change runtime type > GPU (A100)'
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability()
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'{name} | compute capability {major}.{minor} | {total_gb:.1f} GB')
assert major >= 8, f'bf16 autocast needs capability >= 8, this device is {major}.{minor}'

## 2 · Trainable params and FLOPs — measured, not quoted

**The honesty rule, before the numbers.** Report **trainable parameters AND measured FLOPs**.
Never "3.4 M vs 30 M": the frozen ConvNeXt-B trunk runs on every forward pass and dominates the
arithmetic, so a parameter count alone makes this head look ~10× cheaper than it is at inference
time. The comparable figure is GFLOPs at the same input shape.

Two shapes are measured:

* **1024 × 576** — the shape the box memo measured at (**477 GFLOPs**), so the point head's
  number is directly comparable with it. This is the figure the results table carries.
* **960 × 544** — the shape the model actually sees on a Brackish val frame: 960 × 540 padded
  bottom-right to a multiple of 32 (`(-540) % 32 = 4`). Reported because it is the truth about
  this run's inference cost, and it is *not* the comparable number.

The expected trainable count is ≈ 3.36 M (the pyramid decoder). The box head's was 3.58 M, so
the geometry branch this branch removes is ≈ 0.22 M parameters — worth holding on to, because
`4_evaluate` has to say plainly that this experiment cannot separate "0.22 M fewer parameters"
from "two fewer loss terms pulling on the shared fusion trunk".

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
from cropcounter.train import load_checkpoint

device = torch.device('cuda')
models, cfgs = {}, {}
for ck in ('best', 'last'):
    models[ck], cfgs[ck] = load_checkpoint(f'{RUN}/{ck}.pt', device, weights_dir='weights')
    models[ck].eval()
cfg = cfgs['best']   # the run's own config travels inside the checkpoint; decode params come from IT
print(f'decode params from the checkpoint: stride {cfg.output_stride} | k {cfg.k} | '
      f'nms_radius {cfg.nms_radius} | sigma {cfg.sigma} | reference tau {cfg.tau} | '
      f'labels {cfg.labels}')

model = models['best']
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_decoder   = sum(p.numel() for p in model.decoder.parameters())
n_backbone  = sum(p.numel() for p in model.backbone.parameters())

gflops = {}
for (h, w) in ((576, 1024), (544, 960)):
    x = torch.randn(1, 3, h, w, device=device)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        model(x)
    gflops[f'{w}x{h}'] = round(fc.get_total_flops() / 1e9, 1)
    del x

payload = {
    'run': RUN,
    'checkpoint_measured': 'best.pt',
    'trainable_params': int(n_trainable),
    'trainable_params_M': round(n_trainable / 1e6, 3),
    'decoder_params': int(n_decoder),
    'frozen_backbone_params': int(n_backbone),
    'total_params': int(n_trainable + n_backbone),
    'total_params_M': round((n_trainable + n_backbone) / 1e6, 1),
    'gflops': gflops,
    'gflops_comparable_shape': '1024x576',
    'gflops_val_shape': '960x544',
    'note': ('trainable params AND measured GFLOPs are both reported; the frozen '
             'ConvNeXt-B trunk runs on every forward and dominates the FLOPs. '
             'The 1024x576 figure is the one comparable with the box memo.'),
}
json.dump(payload, open(f'{RES}/flops_params.json', 'w'), indent=2)

print(f'trainable (decoder) : {n_trainable:,}  ({n_trainable / 1e6:.2f} M)')
print(f'frozen backbone     : {n_backbone:,}  ({n_backbone / 1e6:.1f} M)')
print(f'total on every pass : {(n_trainable + n_backbone) / 1e6:.1f} M')
for shape, value in gflops.items():
    print(f'GFLOPs @ {shape:<9}: {value}')
print(f"\ncomparable figure: {gflops['1024x576']} GFLOPs at 1024x576 vs the box head's "
      f"measured 477 GFLOPs at the same shape (from the box memo).")
print(f'wrote {RES}/flops_params.json')

## 3 · Whole-val inference — both checkpoints, NMS 1.5 and NMS 5, floor τ 0.05

One forward pass per frame serves everything below: the probability map is decoded four times
on the CPU (two checkpoints × two NMS radii) at the **floor** τ = 0.05. NMS runs inside the
decode and before the threshold, so a higher τ is a mask over these points — that is the whole
reason the floor is low.

`nms_radius` **1.5 is the run's own setting** (`config.json`) and is what the results table
uses; 5 is written alongside purely for the record, because the wheat example's inference
notebook runs at 5 and someone will ask.

`counts_*.csv` are written at the **calibrated** τ read from `2_training`'s
`tau_calibration.json` — never at the config's untuned 0.3. The reader below prints exactly
where each threshold came from; a fallback to the config default is a loud warning, because a
wrong headline τ would invalidate every operating-point number in `4_evaluate`.

In [ ]:
FLOOR_TAU = 0.05          # the decode floor; every sweep threshold is a mask over this set
NMS_RADII = (1.5, 5.0)    # 1.5 = the run's own config; 5 = for the record
HEADLINE_NMS = 1.5

tau, tau_notes = nbh.read_tau_calibration(
    f'{RES}/tau_calibration.json', checkpoints=('best', 'last'),
    nms_radii=NMS_RADII, default_tau=float(cfg.tau),
)
for line in tau_notes:
    print(line)
print('\nheadline operating points:',
      {ck: tau[ck][HEADLINE_NMS] for ck in ('best', 'last')})

In [ ]:
import gc
import numpy as np
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from cropcounter.crop_dataset import CropTileDataset, collate_val, load_splits
from cropcounter.inference import decode_in_bounds, predict_prob, write_cvat_xml

# labels=['fish'] is NOT optional: the default is the wheat example's ("Wheat", "Volunteer"),
# which filters every fish point out and yields a silently empty dataset.
train_recs, val_recs = load_splits(Path(POINTS), fmt='coco', labels=list(cfg.labels))
val_images_dir = Path(POINTS) / 'val' / 'images'
n_gt_points = sum(len(r.points) for r in val_recs)
print(f'{len(train_recs)} train / {len(val_recs)} val records | {n_gt_points} val GT points')

val_ds = CropTileDataset(val_recs, val_images_dir, train=False,
                         output_stride=cfg.output_stride, sigma=cfg.sigma)
loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate_val,
                    num_workers=2, prefetch_factor=2)

# image_preds[(checkpoint, nms_radius)] -> the list write_cvat_xml consumes
image_preds = {(ck, r): [] for ck in ('best', 'last') for r in NMS_RADII}
counts = {ck: [] for ck in ('best', 'last')}
t0 = time.time()
for i, (rec, batch) in enumerate(zip(val_recs, tqdm(loader, desc=f'val ({len(val_recs)} frames)'))):
    assert batch['name'] == rec.name, 'loader order drifted from records'
    for ck in ('best', 'last'):
        prob = predict_prob(models[ck], batch['image'][0], device)
        for radius in NMS_RADII:
            points, scores = decode_in_bounds(
                prob, rec.width, rec.height, tau=FLOOR_TAU, k=cfg.k,
                nms_radius=radius, output_stride=cfg.output_stride,
            )
            image_preds[(ck, radius)].append({
                'name': rec.name, 'width': rec.width, 'height': rec.height,
                'points': points, 'scores': scores, 'label': 'fish',
            })
            if radius == HEADLINE_NMS:
                counts[ck].append({
                    'image_name': rec.name,
                    'predicted_count': int((scores >= tau[ck][HEADLINE_NMS]).sum()),
                })
        del prob
    del batch
    if (i + 1) % 200 == 0:
        gc.collect()
print(f'inference: {time.time() - t0:.0f}s for {len(val_recs)} frames x 2 checkpoints')

XML = {}
for ck in ('best', 'last'):
    for radius in NMS_RADII:
        suffix = '' if radius == HEADLINE_NMS else f'_nms{radius:g}'
        path = f'{RES}/pred_points_{ck}{suffix}.xml'
        write_cvat_xml(image_preds[(ck, radius)], Path(path), label='fish')
        XML[f'point head {ck}' + (f' nms{radius:g}' if suffix else '')] = path
        total = sum(len(p['points']) for p in image_preds[(ck, radius)])
        print(f'{ck}.pt nms {radius}: {total:>7} points at the floor tau {FLOOR_TAU} -> {path}')
    nbh.write_counts_csv(counts[ck], f'{RES}/counts_{ck}.csv')
    kept = np.array([r['predicted_count'] for r in counts[ck]])
    print(f'counts_{ck}.csv @ calibrated tau {tau[ck][HEADLINE_NMS]:.2f}: total {kept.sum()} '
          f'| mean {kept.mean():.3f} | max {kept.max()} | {int((kept > 0).sum())} non-empty frames '
          f'(GT total {n_gt_points})')

## 4 · Comparators — the box systems, through the *same* file shape

The parked box run and the released RF-DETR-Nano already have COCO results on these frames.
Reducing each box to its centre asks them the point question without retraining anything, and
writing them through `point_in_box.coco_results_to_cvat_points` (which goes through
`cropcounter.inference.write_cvat_xml`) means all five systems reach `4_evaluate` as the same
kind of file.

**Two joins have to be right or the score is meaningless:**

1. **Image ids.** The box results are keyed by the CFD **string** id. The points root renumbered
   its images to consecutive ints and kept the original on every record as `cfd_image_id`
   (`cfd_id_map.json` is the same map). The comparator XMLs are therefore written with the
   points root's own `file_name`s, resolved *through* `cfd_image_id`, and the cell asserts the
   map and the records agree — so every XML below is keyed identically to the point head's.
2. **The score floor.** The comparators are filtered to `score >= 0.05`, the same floor the
   point-head XMLs are decoded at. Nothing below 0.05 is ever scored, and RF-DETR's file holds
   938k raw detections at threshold 0.001 — writing them all would produce a ~150 MB XML that
   changes no number. `4_evaluate` cross-checks the resulting rows against the published
   `docs/free_first_step.json`, which is the proof that this filter and this join are lossless
   where it counts.

In [ ]:
BOX_RUN = f'{DRIVE}/runs/brackish_frozen_s0'
COMPARATORS = {
    'box head best (centres)': (f'{BOX_RUN}/predictions.json',      'pred_boxhead_best_centres.xml'),
    'box head last (centres)': (f'{BOX_RUN}/predictions_last.json', 'pred_boxhead_last_centres.xml'),
    'RF-DETR-Nano (centres)':  (f'{DRIVE}/results/rfdetr_nano_640_predictions.json',
                                'pred_rfdetr_nano_centres.xml'),
}

points_doc = json.load(open(f'{POINTS}/val/annotations.json'))
id_map_val = json.load(open(f'{POINTS}/cfd_id_map.json'))['val']   # CFD string id -> points int id

# Image records keyed by the CFD string id the box results use, carrying the POINTS root's
# file_name — so the XML's image names match the point head's, frame for frame.
images_by_cfd_id = {}
for im in points_doc['images']:
    cfd_id = im['cfd_image_id']
    assert int(id_map_val[str(cfd_id)]) == int(im['id']), f'cfd_id_map disagrees for {cfd_id}'
    images_by_cfd_id[cfd_id] = {
        'id': cfd_id, 'file_name': Path(str(im['file_name'])).name,
        'width': int(im['width']), 'height': int(im['height']),
    }
print(f'{len(images_by_cfd_id)} val frames joined through cfd_image_id / cfd_id_map.json')

for label, (src, out_name) in COMPARATORS.items():
    results = json.load(open(src))
    unknown = {d['image_id'] for d in results} - set(images_by_cfd_id)
    assert not unknown, f'{label}: {len(unknown)} predicted image ids are not in the val split'
    kept = [d for d in results if float(d['score']) >= FLOOR_TAU]
    path = f'{RES}/{out_name}'
    pib.coco_results_to_cvat_points(kept, images_by_cfd_id, path, label='fish')
    XML[label] = path
    print(f'{label}: {len(results)} raw -> {len(kept)} at score >= {FLOOR_TAU} '
          f'({len(kept) / max(len(results), 1):.1%}) -> {out_name}')

## 5 · CVAT round-trip sanity

Every file `4_evaluate` reads is parsed back here, before the runtime is released. Two parsers,
because two consumers exist: `cropcounter.crop_dataset.parse_cvat_1_1` (the library's loader —
note `labels=['fish']`, or its wheat default silently drops every point) and
`point_in_box.parse_pred_points` (the scorer's, which divides CVAT's integer percent back to a
0–1 float).

The one lossy step, stated so `4_evaluate` can account for it: CVAT stores confidence as a
**whole percent**, so a score within 0.005 of a sweep threshold can land on the other side of
it after a round trip. That is why the cross-check against `free_first_step.json` in the next
notebook is a tolerance, not an equality.

In [ ]:
from cropcounter.crop_dataset import parse_cvat_1_1

EXPECT_N_IMAGES = 3127   # the Brackish val split
for label, path in XML.items():
    parsed = parse_cvat_1_1(Path(path), labels=['fish'])
    total = sum(len(r.points) for r in parsed)
    assert len(parsed) == EXPECT_N_IMAGES, f'{label}: {len(parsed)} images, expected {EXPECT_N_IMAGES}'
    print(f'{label:<28} {len(parsed)} images | {total:>7} points | {os.path.basename(path)}')

probe = XML['point head best']
parsed_preds = pib.parse_pred_points(probe)
assert len(parsed_preds) == EXPECT_N_IMAGES
arrays = [v[:, 2] for v in parsed_preds.values() if len(v)]
confs = np.concatenate(arrays) if arrays else np.zeros(0)
# Guarded: a head that decoded nothing at all at the 0.05 floor would be a failed run, and
# should say so here rather than die inside np.concatenate on an empty list.
detail = (f'confidence {confs.min():.2f}–{confs.max():.2f} (0-1 floats, as the scorer wants)'
          if len(confs) else '!! NO POINTS AT ALL at the floor tau — check the run')
print(f'\nparse_pred_points({os.path.basename(probe)}): {len(parsed_preds)} images, '
      f'{sum(len(v) for v in parsed_preds.values())} points, {detail}')

## 6 · Visual check — a seeded handful of val frames

Ground-truth points **green**, the point head (`best.pt`, NMS 1.5, at its calibrated τ) **red**.
Six frames drawn from those that contain fish and two empty ones, seeded so a re-run shows the
same frames. The per-frame two-panel figures (`save_visualization`) also go to `viz/` because
the predicted heatmap beside the overlay is what tells a missed fish (no heat) apart from a
suppressed peak (heat, no point).

In [ ]:
from cropcounter.inference import save_visualization

HEADLINE_CK = 'best'
rng = np.random.default_rng(0)
with_fish = [i for i, r in enumerate(val_recs) if r.points]
empty = [i for i, r in enumerate(val_recs) if not r.points]
picks = sorted(rng.choice(with_fish, size=min(6, len(with_fish)), replace=False).tolist()
               + rng.choice(empty, size=min(2, len(empty)), replace=False).tolist())

# A short, UNIQUE file stem for a CFD frame. Brackish file names run ~90 chars and share a long
# clip prefix, so truncating from the FRONT collapses every frame of a clip onto one file — 3,127
# frames would write ~18 images and silently lose the rest. The frame index guarantees
# uniqueness; the tail of the clip-and-frame part keeps it readable.
def viz_stem(index, name):
    return f'{index:05d}_{str(name).split("_jpg.rf.")[0][-40:]}'


frames = []
for idx in picks:
    rec = val_recs[idx]
    item = val_ds[idx]
    prob = predict_prob(models[HEADLINE_CK], item['image'], device)
    points, scores = decode_in_bounds(
        prob, rec.width, rec.height, tau=tau[HEADLINE_CK][HEADLINE_NMS], k=cfg.k,
        nms_radius=HEADLINE_NMS, output_stride=cfg.output_stride,
    )
    save_visualization(item['image'], prob, points, rec.width, rec.height,
                       Path(f'{RES}/viz/{viz_stem(idx, rec.name)}.jpg'),
                       output_stride=cfg.output_stride)
    frames.append({
        'image_path': str(val_images_dir / rec.name),
        'title': f'{rec.name.split("_jpg.rf.")[0][-34:]}\nGT {len(rec.points)} · pred {len(points)}',
        'points': [('GT', np.asarray(item['points']).reshape(-1, 2)),
                   (f'point head {HEADLINE_CK}', points)],
    })
    del prob

grid = nbh.save_overlay_grid(
    frames, Path(f'{RES}/viz/val_samples_{HEADLINE_CK}.png'), ncols=2,
    suptitle=(f'Brackish val, seeded sample — GT green, point head {HEADLINE_CK}.pt red '
              f'at its calibrated tau {tau[HEADLINE_CK][HEADLINE_NMS]:.2f}, NMS {HEADLINE_NMS}'),
)
from IPython.display import Image as IPImage, display
display(IPImage(str(grid)))
print(f'{len(list(Path(f"{RES}/viz").iterdir()))} files in {RES}/viz')

## 7 · Hand-off

Everything of value is on **Drive** already — nothing above writes anything a released runtime
would take with it. `results/points/` now holds the seven prediction XMLs, `counts_best.csv`,
`counts_last.csv`, `flops_params.json` and `viz/`.

`4_evaluate.ipynb` needs no GPU: it reads those XMLs, the bbox ground truth and
`flops_params.json`, and produces the like-for-like table, the Fig-11/Fig-12 analogues and the
spot checks.